# 00 · Run the whole pipeline

Rebuilds the bronze, silver and gold from the four CSVs, using one `run_id` for every step so a run can be traced through `audit.run_log`, `audit.rejects` and `audit.validation_results`.

What each notebook gives: 
| Order | Notebook | Layer | Output |
|---|---|---|---|
| 1 | `10_bronze_ingest` | Bronze | 4 raw tables + file manifest |
| 2 | `20_silver_reference` | Silver | State, keyword and hazard crosswalks |
| 3 | `21_silver_ica` | Silver | `ica_events` |
| 4 | `22_silver_disaster_events` | Silver | `disaster_events`, `disaster_event_states` |
| 5 | `23_silver_abs` | Silver | `abs_state_year` |
| 6 | `24_silver_event_match` | Silver | `ica_de_match` |
| 7 | `30_gold_dim_date` | Gold | `dim_date` |
| 8 | `31_gold_dimensions` | Gold | `dim_location`, `dim_hazard`, `dim_event` |
| 9 | `32_gold_facts` | Gold | `fact_event_loss`, `fact_state_year` |
| 10 | `40_gold_validate` | Audit | Validation + DQ results (stops the run on failure) |
| 11 | `50_gold_analytics` | Gold | OLAP query results and charts |

In [0]:
import uuid
from datetime import datetime
run_id = datetime.now().strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:6]
PIPELINE = ["10_bronze_ingest", "20_silver_reference", "21_silver_ica", "22_silver_disaster_events", "23_silver_abs",
            "24_silver_event_match", "30_gold_dim_date", "31_gold_dimensions", "32_gold_facts", "40_gold_validate", "50_gold_analytics"]
for nb in PIPELINE:
    started = datetime.now()
    print(f"▶ {nb} ...", end=" ")
    dbutils.notebook.run(f"./{nb}", 1800, {"run_id": run_id})
    print(f"done in {(datetime.now() - started).seconds}s")
print("Pipeline finished. run_id =", run_id)

In [0]:
display(spark.sql(f"SELECT layer, step, target, rows_in, rows_out, rows_rejected, status, note FROM workspace.audit.run_log WHERE run_id = '{run_id}' ORDER BY logged_at"))
display(spark.sql(f"SELECT category, check_name, result FROM workspace.audit.validation_results WHERE run_id = '{run_id}'"))